# 03 – Dört Yöntemin Nihai Modelleri, Kilitli Test Tahminleri ve Bağımsız Test
| Bölüm | Ne yapar | Test etiketi? |
|---|---|---|
| **A** | Her yöntemin (A–D) nihai konfigürasyonu tüm development üzerinde **3 başlangıç seed'i** ile eğitilir. LogReg referansı eklenir. **Tek** etiketsiz `test_predictions_locked.csv` dosyası ve SHA-256 manifestosu üretilir. Birincil yöntem ve seed önceden ilan edilir. Şekil 3–4 | **Okunmaz** |
| ⛔ | Kilitli dosya ayrı bir commit ile GitHub'a gönderilir | – |
| **B** | Dört yöntemin test metrikleri, bootstrap %95 GA, seed kararlılığı, confusion matrix, ROC/PR, hata analizi, aşama karşılaştırması, Şekil 1, README ve **rapor taslağı** (`rapor_taslagi.md`) | Okunur |

Test sonuçlarına göre yöntem, model, eşik veya ön işlemede **hiçbir değişiklik yapılmaz**. Ana sonuç, 02'de önceden tanımlı kuralla seçilen birincil yöntemdir.

In [ ]:
# === Gömülü proje dosyaları (src_grup/ ve configs_grup/project.json) – bu hücreyi değiştirmeyin ===
# Kurulum hücresi bu dosyaları Drive'daki proje klasörüne yazar; ayrıca dosya yüklemeye gerek yoktur.
EMBEDDED_FILES = {}
EMBEDDED_FILES['src_grup/__init__.py'] = r''''''
EMBEDDED_FILES['src_grup/ysa_utils.py'] = r'''"""
YSA Ara Sınav Projesi – ortak yardımcı modül
=============================================
grup01_eda, grup02_nested_cv ve grup03_final_test defterlerinin TAMAMI bu dosyadaki
fonksiyonları kullanır. Böylece nested CV'de seçilen model ile nihai testte
eğitilen model birebir aynı kodla (aynı ön işleme, aynı eğitim döngüsü)
üretilir.

İçerik
------
- Seed türetme ve sabitleme
- Kural tabanlı (fit gerektirmeyen) özellik hazırlama: pdays = 999 kodu
- Kat-içi (fold-local) ön işleme: imputer → clipper → scaler | unknown → one-hot
- PyTorch MLP, optimizer, ReduceLROnPlateau + early stopping eğitim döngüsü
- Metrikler, Mean ± SD özetleri ve bootstrap güven aralıkları
- Manuel (M1–M6) ve otomatik (R1–R6) hiperparametre adayları
"""

import copy
import hashlib
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    auc, brier_score_loss, confusion_matrix, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler

# ---------------------------------------------------------------------------
# 0. Sabitler
# ---------------------------------------------------------------------------
TARGET = "y"
ID_COL = "original_index"
GROUP_COL = "dup_group"          # kopya grubu kimliği (duration hariç aynı girdiler)
DROP_COLS = ["duration"]           # Bölüm 2.2: görüşme öncesi bilinmiyor
PDAYS_NO_CONTACT = 999             # Bölüm 2.4: "daha önce temas yok" kodu
CLIP_EXCLUDE = ["pdays", "prev_contacted"]   # Özel kodlu / ikili sütunlar kırpılmaz

TRAINING_SETTINGS = {
    "max_epochs": 150,
    "early_stopping": {"monitor": "internal_val_BCE", "mode": "min",
                       "patience": 15, "min_delta": 1e-6, "restore_best_weights": True},
    "early_stop_subset": {"fraction_of_training_part": 0.15, "stratified": True},
    "lr_scheduler": {"name": "ReduceLROnPlateau", "mode": "min", "factor": 0.5,
                     "patience": 5, "min_lr": 1e-5, "monitor": "internal_val_BCE"},
    "loss": "BCEWithLogitsLoss (tek logit çıkış, olasılık = sigmoid(logit))",
    "selection_metric": "average_precision_score (inner CV ortalama AP)",
    "threshold": 0.50,
    "sgd_momentum": 0.9,
    "clip_quantiles": [0.01, 0.99],
    "device": "cpu",
}


# ---------------------------------------------------------------------------
# 1. Seed yönetimi
# ---------------------------------------------------------------------------
def derive_seeds(student_last5: int) -> dict:
    """Bölüm 3: Temel seed = öğrenci numarasının son 5 hanesi + 2026.
    Tüm alt seed'ler temel seed'den sabit ofsetlerle deterministik türetilir."""
    base = int(student_last5) + 2026
    return {
        "student_last5": int(student_last5),
        "base_seed": base,
        "split_seed": base + 1,
        "outer_cv_seed": base + 2,
        "inner_cv_seed": base + 3,          # dış kat k için: base + 3 + k ; tam dev için: base + 3
        "random_search_seed": base + 4,
        "early_stop_seed": base + 5,
        "bootstrap_seed": base + 6,
        "logreg_seed": base + 7,
        "model_init_seeds": [base + 10, base + 20, base + 30],   # ilki birincil (primary)
        "derivation": "base = last5 + 2026; split=+1, outer=+2, inner=+3(+k), random_search=+4, "
                      "early_stop=+5, bootstrap=+6, logreg=+7, model_init=+10/+20/+30",
    }


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(1)                    # CPU'da toplama sırası sabit → tekrarlanabilirlik
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def file_sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False, default=_json_default)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


# ---------------------------------------------------------------------------
# 2. Kural tabanlı özellik hazırlama (fit YOK → sızıntı riski yok)
# ---------------------------------------------------------------------------
def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """Hedefi ve kimlik sütununu çıkarır, pdays = 999 kodunu iki sütuna ayırır.

    pdays = 999 bir aykırı değer değil, "daha önce temas yok" kodudur (Bölüm 2.4).
    Ham haliyle bırakıldığında (i) sayısal olarak 0–27 gün aralığındaki gerçek
    değerlerden ~40 kat büyük görünür, (ii) gözlemlerin ~%96'sı 999 olduğundan
    IQR = 0 olur ve RobustScaler bu sütunu ölçeklemeden [-999, 0] aralığında
    bırakır. Bu nedenle satır bazlı, veri istatistiği kullanmayan bir kural
    uygulanır:
        prev_contacted = 1[pdays != 999]
        pdays          = pdays  (temas varsa) | 0 (temas yoksa)
    Hiçbir gözlem silinmez; bilgi kaybı yoktur.
    """
    X = df.drop(columns=[c for c in [TARGET, ID_COL, GROUP_COL] + DROP_COLS if c in df.columns]).copy()
    no_contact = X["pdays"] == PDAYS_NO_CONTACT
    X["prev_contacted"] = (~no_contact).astype(int)
    X["pdays"] = np.where(no_contact, 0, X["pdays"]).astype(float)
    return X


def get_feature_lists(X: pd.DataFrame):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    return num_cols, cat_cols


# ---------------------------------------------------------------------------
# 2b. Kopya grubu farkındalıklı bölünmeler
# ---------------------------------------------------------------------------
def duplicate_groups(df: pd.DataFrame, feature_cols) -> pd.Series:
    """duration hariç tüm girdileri aynı olan satırlara ortak grup kimliği verir."""
    return df.groupby(list(feature_cols), sort=False, dropna=False).ngroup().astype(int).rename(GROUP_COL)


def group_aware_holdout(y, groups, test_size, seed):
    """Stratified hold-out; ardından iki tarafa bölünmüş kopya grupları bütünüyle
    eğitim tarafına taşınır. Kopya yoksa sonuç sıradan stratified split ile aynıdır.
    Dönüş: (train_pos, holdout_pos) – konum indeksleri."""
    y = np.asarray(y)
    pos = np.arange(len(y))
    tr, te = train_test_split(pos, test_size=test_size, stratify=y, random_state=seed)
    if groups is not None:
        g = np.asarray(groups)
        straddle = np.intersect1d(g[tr], g[te])
        if len(straddle):
            move = np.isin(g[te], straddle)
            tr = np.concatenate([tr, te[move]])
            te = te[~move]
    return np.sort(tr), np.sort(te)


def make_cv_splits(y, groups, n_splits, seed):
    """Kopya yoksa StratifiedKFold; varsa aynı grup tek kata düşsün diye StratifiedGroupKFold."""
    y = np.asarray(y)
    dummy = np.zeros(len(y))
    if groups is None or len(np.unique(groups)) == len(y):
        return list(StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(dummy, y))
    return list(StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed).split(dummy, y, np.asarray(groups)))


# ---------------------------------------------------------------------------
# 2c. Önbellek (Colab bağlantısı koparsa kaldığı yerden devam)
# ---------------------------------------------------------------------------
RESULT_KEYS = ["best_epoch", "stopped_epoch", "best_es_loss", "status", "warnings", "n_lr_drops",
               "final_lr", "n_params", "input_dim", "n_fit", "n_es", "n_es_pos", "elapsed_sec", "pos_weight_used"]


def index_hash(index) -> str:
    return hashlib.sha1(np.asarray(index).astype(np.int64).tobytes()).hexdigest()[:12]


def run_hash(cfg, extra) -> str:
    keys = [k for k in cfg if k not in ("purpose",)]
    blob = json.dumps({"cfg": {k: cfg[k] for k in sorted(keys)}, "extra": extra, "settings": TRAINING_SETTINGS},
                      sort_keys=True, default=_json_default)
    return hashlib.sha1(blob.encode()).hexdigest()[:16]


def run_cached(key, cfg, extra, fn, cache_dir="results_grup/cache"):
    """fn() JSON'a yazılabilir bir sözlük döndürmeli. Aynı anahtar + aynı hash varsa yeniden eğitilmez."""
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"{key}.json")
    h = run_hash(cfg, extra)
    if os.path.exists(path):
        try:
            rec = load_json(path)
            if rec.get("hash") == h:
                rec["from_cache"] = True
                return rec
        except Exception:
            pass
    rec = fn()
    rec["hash"] = h
    save_json(rec, path)
    rec["from_cache"] = False
    return rec


def result_record(res, eval_name) -> dict:
    """train_mlp çıktısından JSON'a yazılabilir kayıt (olasılıklar + geçmiş + bilgi)."""
    rec = {k: res[k] for k in RESULT_KEYS}
    rec["probs"] = [float(v) for v in res["probs"][eval_name]]
    rec["history"] = res["history"]
    return rec


# ---------------------------------------------------------------------------
# 3. Kat-içi ön işleme bileşenleri
# ---------------------------------------------------------------------------
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Eğitim alt kümesinde öğrenilen yüzdelik sınırlarla kırpma (winsorizing).

    NOT: Pipeline içinde SimpleImputer numpy dizisi döndürdüğü için sütun ADI
    değil sütun İNDEKSİ ile çalışır (eski sürümdeki "pdays muafiyeti çalışmıyor"
    hatası bu şekilde giderildi).
    """

    def __init__(self, lower_q=0.01, upper_q=0.99, enabled=False, exclude_idx=()):
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.enabled = enabled
        self.exclude_idx = exclude_idx

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n = X.shape[1]
        self.n_features_in_ = n
        self.lower_ = np.full(n, -np.inf)
        self.upper_ = np.full(n, np.inf)
        if self.enabled:
            excl = set(self.exclude_idx)
            cols = [j for j in range(n) if j not in excl]
            if cols:
                self.lower_[cols] = np.quantile(X[:, cols], self.lower_q, axis=0)
                self.upper_[cols] = np.quantile(X[:, cols], self.upper_q, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)


class UnknownHandler(BaseEstimator, TransformerMixin):
    """'unknown' kategorisi için iki seçenek (Bölüm 2.3):
    - keep_category : 'unknown' ayrı bir kategori olarak kalır
    - most_frequent : 'unknown' → NaN, ardından kat-içi en sık sınıf ile doldurulur
    """

    def __init__(self, strategy="keep_category"):
        self.strategy = strategy

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).astype(object).copy()
        if self.strategy == "most_frequent":
            X_df = X_df.replace("unknown", np.nan)
        return X_df


def make_scaler(name: str):
    if name == "StandardScaler":
        return StandardScaler()
    if name == "MinMaxScaler":
        return MinMaxScaler()
    if name == "RobustScaler":
        return RobustScaler()
    raise ValueError(f"Bilinmeyen scaler: {name}")


def build_preprocessor(num_cols, cat_cols, scaler_name="StandardScaler",
                       clip_outliers=False, unknown_strategy="keep_category"):
    exclude_idx = tuple(num_cols.index(c) for c in CLIP_EXCLUDE if c in num_cols)
    q_lo, q_hi = TRAINING_SETTINGS["clip_quantiles"]
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clipper", QuantileClipper(q_lo, q_hi, enabled=clip_outliers, exclude_idx=exclude_idx)),
        ("scaler", make_scaler(scaler_name)),
    ])
    cat_steps = [("unk_handler", UnknownHandler(strategy=unknown_strategy))]
    if unknown_strategy == "most_frequent":
        cat_steps.append(("imputer", SimpleImputer(strategy="most_frequent")))
    cat_steps.append(("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)))
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", Pipeline(cat_steps), cat_cols),
    ])


def get_learned_stats(preprocessor, num_cols, cat_cols) -> dict:
    """Fit edilmiş ön işlemcinin öğrendiği istatistikleri JSON'a yazılabilir hale getirir
    (Bölüm 12.3: öğrenilmiş dönüşüm istatistikleri saklanacak)."""
    num_pipe = preprocessor.named_transformers_["num"]
    cat_pipe = preprocessor.named_transformers_["cat"]
    scaler = num_pipe.named_steps["scaler"]
    clipper = num_pipe.named_steps["clipper"]
    stats = {
        "numeric_columns": num_cols,
        "imputer_median": dict(zip(num_cols, num_pipe.named_steps["imputer"].statistics_.tolist())),
        "clip_enabled": bool(clipper.enabled),
        "clip_lower": dict(zip(num_cols, [None if np.isinf(v) else float(v) for v in clipper.lower_])),
        "clip_upper": dict(zip(num_cols, [None if np.isinf(v) else float(v) for v in clipper.upper_])),
        "scaler": type(scaler).__name__,
    }
    for attr in ["mean_", "scale_", "center_", "data_min_", "data_max_"]:
        if hasattr(scaler, attr) and getattr(scaler, attr) is not None:
            stats[f"scaler_{attr.rstrip('_')}"] = dict(zip(num_cols, getattr(scaler, attr).tolist()))
    onehot = cat_pipe.named_steps["onehot"]
    stats["onehot_categories"] = {c: [str(v) for v in cats] for c, cats in zip(cat_cols, onehot.categories_)}
    if "imputer" in cat_pipe.named_steps:
        stats["cat_imputer_most_frequent"] = dict(
            zip(cat_cols, [str(v) for v in cat_pipe.named_steps["imputer"].statistics_]))
    stats["n_output_features"] = int(len(num_cols) + sum(len(c) for c in onehot.categories_))
    return stats


def apply_pdays_mode(X_train, eval_sets, num_cols, cfg):
    """cfg['pdays_mode'] == 'raw' ise prepare_features'ın pdays kuralını geri alır:
    pdays = 999 kodu ham haliyle kullanılır, prev_contacted göstergesi çıkarılır (Yöntem D)."""
    if cfg.get("pdays_mode", "indicator") != "raw":
        return X_train, eval_sets, num_cols
    def raw(X):
        X = X.copy()
        X["pdays"] = np.where(X["prev_contacted"] == 0, PDAYS_NO_CONTACT, X["pdays"]).astype(float)
        return X.drop(columns=["prev_contacted"])
    return raw(X_train), {k: raw(v) for k, v in eval_sets.items()}, [c for c in num_cols if c != "prev_contacted"]


def resolve_pos_weight(cfg, y_fit):
    """pos_weight: None/1 → ağırlıksız; 'auto' → fit alt kümesindeki neg/pos oranı (kat-içi); sayı → o değer."""
    pw = cfg.get("pos_weight")
    if pw == "auto":
        y = np.asarray(y_fit)
        return float((y == 0).sum() / max(1, (y == 1).sum()))
    return None if pw in (None, 1, 1.0) else float(pw)


# ---------------------------------------------------------------------------
# 4. MLP modeli ve eğitim döngüsü
# ---------------------------------------------------------------------------
class MLPNet(nn.Module):
    def __init__(self, input_dim, hidden_layers, activation="ReLU", dropout=0.0):
        super().__init__()
        act_cls = nn.ReLU if activation == "ReLU" else nn.Tanh
        layers, in_d = [], input_dim
        for h in hidden_layers:
            layers += [nn.Linear(in_d, h), act_cls()]
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            in_d = h
        layers.append(nn.Linear(in_d, 1))           # tek logit
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def count_mlp_params(input_dim, hidden_layers) -> int:
    """Analitik parametre sayısı: Σ (girdi × çıktı + bias)."""
    dims = [input_dim] + list(hidden_layers) + [1]
    return int(sum(dims[i] * dims[i + 1] + dims[i + 1] for i in range(len(dims) - 1)))


def make_optimizer(model, cfg):
    opt = cfg["optimizer"]
    if opt == "AdamW":
        return torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if opt == "Adam":
        return torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if opt == "SGD":
        return torch.optim.SGD(model.parameters(), lr=cfg["lr"],
                               momentum=TRAINING_SETTINGS["sgd_momentum"],
                               weight_decay=cfg["weight_decay"])
    raise ValueError(f"Bilinmeyen optimizer: {opt}")


def train_mlp(X_train_raw, y_train, eval_sets, cfg, num_cols, cat_cols,
              model_seed, es_seed, max_epochs=None, groups=None):
    """Tek bir MLP eğitimi (iç CV, dış kat refit ve nihai eğitim için ORTAK fonksiyon).

    X_train_raw / y_train : ağırlık eğitimi için kullanılan bölüm (ör. inner-train)
    eval_sets             : {"ad": X_raw} → yalnızca tahmin üretilecek kümeler
                            (inner validation, outer evaluation veya test).
                            Bu kümeler hiçbir fit/early-stop/scheduler kararında kullanılmaz.

    Üçlü ayrım (Bölüm 5):
      (i)  X_train_raw → %85 "fit" alt kümesi : ön işleme fit + ağırlık güncelleme
      (ii) X_train_raw → %15 "early-stop" alt kümesi : early stopping + scheduler (BCE)
      (iii) eval_sets : yalnızca transform + tahmin
    groups : kopya grubu kimlikleri; verilirse aynı kopya hem fit hem early-stop
             alt kümesine düşmez.
    """

    t0 = time.time()
    max_epochs = max_epochs or TRAINING_SETTINGS["max_epochs"]
    es_cfg = TRAINING_SETTINGS["early_stopping"]
    y_train = pd.Series(np.asarray(y_train), index=X_train_raw.index)
    X_train_raw, eval_sets, num_cols = apply_pdays_mode(X_train_raw, eval_sets, num_cols, cfg)

    fit_pos, es_pos = group_aware_holdout(
        y_train.values, groups,
        TRAINING_SETTINGS["early_stop_subset"]["fraction_of_training_part"], es_seed)
    X_fit_raw, X_es_raw = X_train_raw.iloc[fit_pos], X_train_raw.iloc[es_pos]
    y_fit, y_es = y_train.iloc[fit_pos], y_train.iloc[es_pos]

    pre = build_preprocessor(num_cols, cat_cols, cfg["scaler"], cfg["clip_outliers"], cfg["unknown_strategy"])
    X_fit = pre.fit_transform(X_fit_raw)          # FIT yalnızca burada
    X_es = pre.transform(X_es_raw)
    X_eval = {k: pre.transform(v) for k, v in eval_sets.items()}

    seed_everything(model_seed)
    device = torch.device(TRAINING_SETTINGS["device"])
    model = MLPNet(X_fit.shape[1], cfg["hidden_layers"], cfg["activation"], cfg["dropout"]).to(device)
    criterion = nn.BCEWithLogitsLoss()            # early-stop / scheduler: ağırlıksız BCE (Bölüm 8)
    pw = resolve_pos_weight(cfg, y_fit)           # opsiyonel sınıf ağırlığı (yalnızca eğitim kaybı)
    train_criterion = (nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pw), dtype=torch.float32))
                       if pw is not None else criterion)
    optimizer = make_optimizer(model, cfg)
    sch_cfg = TRAINING_SETTINGS["lr_scheduler"]
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode=sch_cfg["mode"], factor=sch_cfg["factor"],
        patience=sch_cfg["patience"], min_lr=sch_cfg["min_lr"])
        if cfg.get("use_scheduler", True) else None)

    to_t = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32, device=device)
    X_fit_t, y_fit_t = to_t(X_fit), to_t(y_fit.values)
    X_es_t, y_es_t = to_t(X_es), to_t(y_es.values)

    g = torch.Generator(device="cpu")
    g.manual_seed(model_seed)
    n, bs = X_fit_t.shape[0], int(cfg["batch_size"])

    best_loss, best_epoch, bad_epochs = float("inf"), 0, 0
    best_state = copy.deepcopy(model.state_dict())
    history, status, warnings_ = [], "max_epochs_reached", []
    stopped_epoch = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=g)
        tot, nb = 0.0, 0
        nan_hit = False
        for s in range(0, n, bs):
            idx = perm[s:s + bs]
            optimizer.zero_grad(set_to_none=True)
            loss = train_criterion(model(X_fit_t[idx]), y_fit_t[idx])
            if not torch.isfinite(loss):
                nan_hit = True
                break
            loss.backward()
            optimizer.step()
            tot += loss.item()
            nb += 1
        if nan_hit:
            status = "nan_loss"
            warnings_.append(f"epoch {epoch}: NaN/Inf kayıp")
            stopped_epoch = epoch
            break

        model.eval()
        with torch.no_grad():
            es_loss = float(criterion(model(X_es_t), y_es_t).item())
        lr_now = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch, "train_loss": tot / max(nb, 1),
                        "es_val_loss": es_loss, "lr": lr_now})
        stopped_epoch = epoch

        if scheduler is not None:
            scheduler.step(es_loss)

        if es_loss < best_loss - es_cfg["min_delta"]:
            best_loss, best_epoch, bad_epochs = es_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= es_cfg["patience"]:
                status = "early_stopped"
                break

    if best_epoch > 0:
        model.load_state_dict(best_state)
    if best_epoch <= 2 and status != "nan_loss":
        warnings_.append(f"en iyi epoch = {best_epoch} (çok erken; hızlı aşırı öğrenme veya yüksek LR olabilir)")
    if status == "max_epochs_reached":
        warnings_.append("150 epoch üst sınırına ulaşıldı (yakınsama tamamlanmamış olabilir)")
    n_lr_drops = int(sum(1 for i in range(1, len(history)) if history[i]["lr"] < history[i - 1]["lr"]))

    model.eval()
    probs = {}
    with torch.no_grad():
        for k, Xa in X_eval.items():
            probs[k] = torch.sigmoid(model(to_t(Xa))).cpu().numpy()

    return {
        "probs": probs, "history": history, "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch, "best_es_loss": best_loss, "status": status,
        "warnings": "; ".join(warnings_), "n_lr_drops": n_lr_drops,
        "final_lr": history[-1]["lr"] if history else cfg["lr"],
        "n_params": model.count_parameters(), "input_dim": int(X_fit.shape[1]),
        "pos_weight_used": float(pw) if pw not in (None,) else 1.0,
        "n_fit": int(len(y_fit)), "n_es": int(len(y_es)),
        "n_es_pos": int(y_es.sum()), "elapsed_sec": time.time() - t0,
        "model": model, "preprocessor": pre,
    }


def history_frame(result, **tags) -> pd.DataFrame:
    """Eğitim geçmişini etiketlerle birlikte DataFrame'e çevirir (CSV loglama için)."""
    h = pd.DataFrame(result["history"])
    for k, v in tags.items():
        h[k] = v
    return h


# ---------------------------------------------------------------------------
# 5. Metrikler
# ---------------------------------------------------------------------------
METRIC_COLS = ["AP", "ROC_AUC", "F1", "Sensitivity", "Specificity",
               "Balanced_Accuracy", "Accuracy", "Precision", "Brier", "PR_AUC_trapz"]


def compute_metrics(y_true, y_prob, threshold=0.50) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "AP": float(average_precision_score(y_true, y_prob)),
        "ROC_AUC": float(roc_auc_score(y_true, y_prob)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Sensitivity": float(recall_score(y_true, y_pred, zero_division=0)),
        "Specificity": float(tn / (tn + fp)) if (tn + fp) else 0.0,
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Brier": float(brier_score_loss(y_true, y_prob)),
        # Trapez PR-AUC: AP ile AYNI şey değildir (Bölüm 11), ayrı raporlanır
        "PR_AUC_trapz": float(auc(*precision_recall_curve(y_true, y_prob)[1::-1])),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "threshold": float(threshold),
    }


def mean_sd_table(df, group_col, metrics=METRIC_COLS) -> pd.DataFrame:
    """Kat sonuçlarından Mean ± örnek SD (ddof=1) tablosu üretir."""
    rows = []
    for g, sub in df.groupby(group_col, sort=False):
        row = {group_col: g, "n": len(sub)}
        for m in metrics:
            vals = sub[m].astype(float).values
            mu = float(np.mean(vals))
            sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else float("nan")
            row[f"{m}_mean"], row[f"{m}_sd"] = mu, sd
            row[m] = f"{mu:.4f} ± {sd:.4f}"
        rows.append(row)
    return pd.DataFrame(rows)


def bootstrap_ci(y_true, y_prob, n_boot=1000, seed=0, threshold=0.50, alpha=0.05):
    """Yüzdelik bootstrap %95 GA (AP, ROC-AUC, F1). Tek sınıf içeren yeniden
    örneklemeler REDDEDİLİR ve yeniden çekilir (Bölüm 13.1)."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = {"AP": [], "ROC_AUC": [], "F1": []}
    rejected = 0
    while len(vals["AP"]) < n_boot:
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        if yt.min() == yt.max():
            rejected += 1
            continue
        yp = y_prob[idx]
        vals["AP"].append(average_precision_score(yt, yp))
        vals["ROC_AUC"].append(roc_auc_score(yt, yp))
        vals["F1"].append(f1_score(yt, (yp >= threshold).astype(int), zero_division=0))
    point = compute_metrics(y_true, y_prob, threshold)
    out = {}
    for m, v in vals.items():
        v = np.asarray(v)
        out[m] = {"point": point[m],
                  "ci_low": float(np.percentile(v, 100 * alpha / 2)),
                  "ci_high": float(np.percentile(v, 100 * (1 - alpha / 2))),
                  "boot_mean": float(v.mean()), "boot_sd": float(v.std(ddof=1))}
    return out, rejected


def f1_optimal_threshold(y_true, y_prob) -> float:
    """İç CV (out-of-fold) tahminlerinde F1'i en yüksek yapan eşik. YALNIZCA development
    tahminleriyle çağrılır; test verisiyle asla."""
    prec, rec, thr = precision_recall_curve(np.asarray(y_true).astype(int), np.asarray(y_prob, float))
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[int(np.nanargmax(f1))])


# ---------------------------------------------------------------------------
# 6. Hiperparametre adayları
# ---------------------------------------------------------------------------
_M_COMMON = {"activation": "ReLU", "batch_size": 64, "dropout": 0.0, "weight_decay": 1e-4,
             "clip_outliers": False, "unknown_strategy": "keep_category"}


def _m(cid, purpose, **kw):
    base = {"config_id": cid, "search_type": "manual", "optimizer": "AdamW", "hidden_layers": [32],
            "lr": 1e-3, "scaler": "StandardScaler", "use_scheduler": True, **_M_COMMON}
    base.update(kw)
    base["purpose"] = purpose
    return base


MANUAL_CONFIGS = [
    _m("M1", "Temel manuel konfigürasyon"),
    _m("M2", "Optimizer etkisi (AdamW → Adam)", optimizer="Adam"),
    _m("M3", "Optimizer etkisi (AdamW → SGD+Momentum 0.9)", optimizer="SGD"),
    _m("M4", "Ölçekleme etkisi (Standard → MinMax)", scaler="MinMaxScaler"),
    _m("M5", "Mimari etkisi ((32,) → (64,32))", hidden_layers=[64, 32]),
    _m("M6", "LR scheduler etkisi (scheduler kapalı)", use_scheduler=False),
]

SEARCH_SPACE = {
    "optimizer": ["SGD", "Adam", "AdamW"],
    "candidates_per_optimizer": 2,
    "hidden_layers": [[16], [32], [64], [64, 32], [128, 64]],
    "activation": ["ReLU", "tanh"],
    "lr_log_uniform": {"Adam": [1e-4, 1e-2], "AdamW": [1e-4, 1e-2], "SGD": [1e-3, 1e-1]},
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3],
    "batch_size": [32, 64, 128],
    "scaler": ["StandardScaler", "MinMaxScaler", "RobustScaler"],
    "dropout": [0.0, 0.2],
    "clip_outliers": [False, True],
    "unknown_strategy": ["keep_category", "most_frequent"],
    "use_scheduler": [True],
}


def make_random_configs(seed: int) -> list:
    """Tekrarlanabilir Random Search: her optimizer için tam 2 aday (toplam 6).
    LR log-uniform örneklenir; SGD için daha geniş aralık (Bölüm 9–10)."""
    rng = np.random.default_rng(seed)
    S = SEARCH_SPACE
    configs, k = [], 1
    for opt in S["optimizer"]:
        for _ in range(S["candidates_per_optimizer"]):
            lo, hi = S["lr_log_uniform"][opt]
            lr = float(10 ** rng.uniform(np.log10(lo), np.log10(hi)))
            configs.append({
                "config_id": f"R{k}", "search_type": "random", "optimizer": opt,
                "hidden_layers": list(S["hidden_layers"][int(rng.integers(len(S["hidden_layers"])))]),
                "activation": str(rng.choice(S["activation"])),
                "lr": round(lr, 6),
                "weight_decay": float(rng.choice(S["weight_decay"])),
                "batch_size": int(rng.choice(S["batch_size"])),
                "scaler": str(rng.choice(S["scaler"])),
                "dropout": float(rng.choice(S["dropout"])),
                "use_scheduler": True,
                "clip_outliers": bool(rng.choice(S["clip_outliers"])),
                "unknown_strategy": str(rng.choice(S["unknown_strategy"])),
                "purpose": "Random Search adayı",
            })
            k += 1
    return configs


def config_label(cfg) -> str:
    return (f"{cfg['optimizer']}, {tuple(cfg['hidden_layers'])}, {cfg['activation']}, "
            f"lr={cfg['lr']:g}, wd={cfg['weight_decay']:g}, bs={cfg['batch_size']}, "
            f"{cfg['scaler'].replace('Scaler', '')}, do={cfg['dropout']}, "
            f"clip={'E' if cfg['clip_outliers'] else 'H'}, unk={cfg['unknown_strategy']}, "
            f"sch={'açık' if cfg.get('use_scheduler', True) else 'kapalı'}")
'''
EMBEDDED_FILES['src_grup/group_methods.py'] = r'''"""
Grup projesi – dört yöntem ailesi (A–D) için ortak nested CV boru hattı
=======================================================================
Her yöntem aynı veri, aynı seed, aynı dış/iç katlar ve aynı 6 manuel + 6 otomatik
aday listesiyle **baştan sona** çalıştırılır. Yöntemler arasındaki tek fark
METHODS sözlüğündeki yöntem-düzeyi kuraldır:

  A  Standart (yönerge akışı)               – değişiklik yok, eşik 0.50
  B  Sınıf ağırlıklı kayıp                  – BCE pos_weight = kat-içi neg/pos
  C  İç CV'de seçilen karar eşiği            – A ile aynı eğitim; eşik iç CV out-of-fold
                                               tahminlerinde F1-optimal olarak seçilir
  D  Alternatif ön işleme                   – unknown → en sık sınıf, P1–P99 clipping,
                                               pdays = 999 ham kodlama (gösterge yok)
"""

import time

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

from .ysa_utils import (
    MANUAL_CONFIGS, compute_metrics, build_preprocessor, f1_optimal_threshold, history_frame,
    index_hash, make_cv_splits, make_random_configs, result_record, run_cached, train_mlp,
)

METHODS = {
    "A": {"name": "Standart (yönerge akışı)", "override": {}, "threshold_mode": "fixed",
          "training_from": "A",
          "how": "Adaylar yönergede tanımlandığı gibi eğitilir: ağırlıksız BCE, `unknown` ayrı kategori, "
                 "M1–M6'da kırpma yok, `pdays = 999` → `prev_contacted` göstergesi + 0. Karar eşiği 0.50.",
          "hypothesis": "Yönergenin temel akışı, diğer yöntemlerin karşılaştırılacağı referans performansı verir."},
    "B": {"name": "Sınıf ağırlıklı kayıp", "override": {"pos_weight": "auto"}, "threshold_mode": "fixed",
          "training_from": "B",
          "how": "Her eğitimde eğitim kaybı `BCEWithLogitsLoss(pos_weight = neg/pos)` olur. Oran, o eğitimin fit alt "
                 "kümesinden hesaplanır (kat-içi, sızıntısız). Early stopping ve scheduler ağırlıksız BCE'yi izler. Eşik 0.50.",
          "hypothesis": "Azınlık sınıfa daha fazla ağırlık vermek 0.50 eşiğinde Sensitivity'yi belirgin artırır, "
                        "sıralama metriği AP'yi az değiştirir, olasılık kalibrasyonunu (Brier) bozar."},
    "C": {"name": "İç CV'de seçilen karar eşiği", "override": {}, "threshold_mode": "inner_oof_f1",
          "training_from": "A",
          "how": "Model eğitimi ve aday seçimi Yöntem A ile aynıdır (aynı ağırlıklar). Fark karar kuralındadır: "
                 "seçilen adayın iç CV out-of-fold tahminlerinde F1'i en yüksek yapan eşik seçilir ve dış değerlendirme "
                 "ile testte bu eşik kullanılır. Eşik hiçbir zaman değerlendirme/test verisiyle seçilmez.",
          "hypothesis": "Dengesiz veride 0.50 eşiği gereğinden yüksektir. İç CV ile seçilen eşik AP'yi değiştirmeden "
                        "F1, Sensitivity ve Balanced Accuracy'yi artırır."},
    "D": {"name": "Alternatif ön işleme", "override": {"unknown_strategy": "most_frequent", "clip_outliers": True,
                                                       "pdays_mode": "raw"},
          "threshold_mode": "fixed", "training_from": "D",
          "how": "Tüm adaylarda `unknown` kategorileri kat-içi en sık sınıfla doldurulur, sayısal değişkenler eğitim katında "
                 "öğrenilen P1–P99 sınırlarıyla kırpılır, `pdays` ham kodlamasıyla (999 dahil) kullanılır ve "
                 "`prev_contacted` göstergesi eklenmez. Diğer her şey A ile aynıdır. Eşik 0.50.",
          "hypothesis": "`unknown`'ı bilgi taşıyan ayrı bir kategori olarak tutmak ve `pdays` özel kodunu ayrı göstermek "
                        "bilgi kaybını önler. Bu yüzden D'nin AP'si A'dan eşit veya düşük olur."},
}
METHOD_ORDER = ["A", "B", "C", "D"]


def method_candidates(mid, seeds):
    """Ortak 6 manuel + 6 otomatik aday listesine yöntem-düzeyi kuralı uygular."""
    base = MANUAL_CONFIGS + make_random_configs(seeds["random_search_seed"])
    tm = METHODS[mid]["training_from"]
    return [dict(c, **METHODS[tm]["override"]) for c in base]


def _run(key, X_tr, y_tr, g_tr, X_ev, cfg, num_cols, cat_cols, model_seed, es_seed):
    extra = {"model_seed": model_seed, "es_seed": es_seed,
             "train_idx": index_hash(X_tr.index), "eval_idx": index_hash(X_ev.index)}
    def fn():
        res = train_mlp(X_tr, y_tr, {"eval": X_ev}, cfg, num_cols, cat_cols,
                        model_seed=model_seed, es_seed=es_seed, groups=g_tr)
        return result_record(res, "eval")
    return run_cached(key, cfg, extra, fn)


def _select(df, stype):
    s = (df[df.search_type == stype].groupby("config_id", sort=False)["inner_AP"]
         .agg(mean="mean", sd="std").reset_index())
    s["_o"] = range(len(s))
    s = s.sort_values(["mean", "sd", "_o"], ascending=[False, True, True])
    return s.iloc[0]


# ---------------------------------------------------------------------------
def run_method_nested(mid, X_dev, y_dev, g_dev, seeds, num_cols, cat_cols, verbose=True):
    """3×3 nested CV: 108 iç eğitim + 6 dış refit (aynı katlar tüm yöntemlerde)."""
    meth = METHODS[mid]
    tm = meth["training_from"]
    cands = method_candidates(mid, seeds)
    y_dev = pd.Series(np.asarray(y_dev), index=X_dev.index)
    g_dev = np.asarray(g_dev)
    outer_splits = make_cv_splits(y_dev, g_dev, 3, seeds["outer_cv_seed"])
    inner_rows, outer_rows, sel_rows, logs = [], [], [], []
    t0 = time.time()
    for k, (otr, oev) in enumerate(outer_splits, start=1):
        X_otr, y_otr, g_otr = X_dev.iloc[otr], y_dev.iloc[otr], g_dev[otr]
        X_oev, y_oev = X_dev.iloc[oev], y_dev.iloc[oev]
        inner_splits = make_cv_splits(y_otr, g_otr, 3, seeds["inner_cv_seed"] + k)
        oof = {c["config_id"]: np.full(len(otr), np.nan) for c in cands}
        for cfg in cands:
            for j, (itr, iva) in enumerate(inner_splits, start=1):
                rec = _run(f"{tm}_outer{k}_inner{j}_{cfg['config_id']}", X_otr.iloc[itr], y_otr.iloc[itr], g_otr[itr],
                           X_otr.iloc[iva], cfg, num_cols, cat_cols,
                           seeds["model_init_seeds"][0] + 100 * k + j, seeds["early_stop_seed"] + 100 * k + j)
                oof[cfg["config_id"]][iva] = rec["probs"]
                inner_rows.append({"method": mid, "outer_fold": k, "inner_fold": j, "config_id": cfg["config_id"],
                                   "search_type": cfg["search_type"], "optimizer": cfg["optimizer"],
                                   "inner_AP": float(average_precision_score(y_otr.iloc[iva], rec["probs"])),
                                   **{x: rec[x] for x in ["best_epoch", "stopped_epoch", "n_lr_drops", "n_params",
                                                          "input_dim", "elapsed_sec", "status", "warnings",
                                                          "pos_weight_used"]},
                                   "from_cache": rec["from_cache"]})
                logs.append(history_frame(rec, method=mid, phase="nested_inner", outer_fold=k, inner_fold=j,
                                          config_id=cfg["config_id"]))
        dfk = pd.DataFrame([r for r in inner_rows if r["outer_fold"] == k])
        for label, stype in [("manual", "manual"), ("random", "random")]:
            best = _select(dfk, stype)
            cid = best["config_id"]
            cfg = next(c for c in cands if c["config_id"] == cid)
            thr = (f1_optimal_threshold(y_otr, oof[cid]) if meth["threshold_mode"] == "inner_oof_f1" else 0.50)
            rec = _run(f"{tm}_outer{k}_refit_{label}_{cid}", X_otr, y_otr, g_otr, X_oev, cfg, num_cols, cat_cols,
                       seeds["model_init_seeds"][0] + 1000 * k, seeds["early_stop_seed"] + 1000 * k)
            m = compute_metrics(y_oev, rec["probs"], thr)
            outer_rows.append({"method": mid, "search": label, "outer_fold": k, "selected_config": cid,
                               "inner_mean_AP": float(best["mean"]), "inner_sd_AP": float(best["sd"]),
                               "best_epoch": rec["best_epoch"], "elapsed_sec": rec["elapsed_sec"], **m})
            sel_rows.append({"method": mid, "outer_fold": k, "search": label, "selected_config": cid,
                             "inner_mean_AP": float(best["mean"]), "threshold": thr})
            logs.append(history_frame(rec, method=mid, phase="outer_refit", outer_fold=k, inner_fold=0,
                                      config_id=cid))
        if verbose:
            sel = ", ".join(f"{r['search']}→{r['selected_config']}" for r in sel_rows if r["outer_fold"] == k)
            print(f"[{mid}] dış kat {k} tamam ({time.time() - t0:.0f} sn) | seçim: {sel}")
    inner = pd.DataFrame(inner_rows)
    assert len(inner) == 108, len(inner)
    return {"inner": inner, "outer": pd.DataFrame(outer_rows), "selection": pd.DataFrame(sel_rows),
            "logs": pd.concat(logs, ignore_index=True)}


def run_references_nested(X_dev, y_dev, g_dev, seeds, num_cols, cat_cols):
    """Dummy ve Logistic Regression: aynı dış katlarda outer-train → outer-eval."""
    y_dev = pd.Series(np.asarray(y_dev), index=X_dev.index)
    rows = []
    for k, (otr, oev) in enumerate(make_cv_splits(y_dev, g_dev, 3, seeds["outer_cv_seed"]), start=1):
        X_otr, y_otr, X_oev, y_oev = X_dev.iloc[otr], y_dev.iloc[otr], X_dev.iloc[oev], y_dev.iloc[oev]
        p = DummyClassifier(strategy="most_frequent").fit(X_otr, y_otr).predict_proba(X_oev)[:, 1]
        rows.append({"method": "Dummy", "search": "-", "outer_fold": k, "selected_config": "most_frequent",
                     **compute_metrics(y_oev, p)})
        p = fit_logreg(X_otr, y_otr, X_oev, seeds, num_cols, cat_cols)
        rows.append({"method": "LogReg", "search": "-", "outer_fold": k, "selected_config": "L2, C=1",
                     **compute_metrics(y_oev, p)})
    return pd.DataFrame(rows)


def fit_logreg(X_tr, y_tr, X_ev, seeds, num_cols, cat_cols):
    pre = build_preprocessor(num_cols, cat_cols, "StandardScaler", False, "keep_category")
    lr = LogisticRegression(max_iter=2000, random_state=seeds["logreg_seed"])
    lr.fit(pre.fit_transform(X_tr), y_tr)
    return lr.predict_proba(pre.transform(X_ev))[:, 1]


# ---------------------------------------------------------------------------
def run_method_fulldev(mid, X_dev, y_dev, g_dev, seeds, num_cols, cat_cols):
    """Tam development 3-kat iç CV (36 eğitim) → yöntemin nihai konfigürasyonu ve eşiği."""
    meth = METHODS[mid]
    tm = meth["training_from"]
    cands = method_candidates(mid, seeds)
    y_dev = pd.Series(np.asarray(y_dev), index=X_dev.index)
    g_dev = np.asarray(g_dev)
    splits = make_cv_splits(y_dev, g_dev, 3, seeds["inner_cv_seed"])
    rows, logs = [], []
    oof = {c["config_id"]: np.full(len(y_dev), np.nan) for c in cands}
    for cfg in cands:
        for j, (itr, iva) in enumerate(splits, start=1):
            rec = _run(f"{tm}_fulldev_inner{j}_{cfg['config_id']}", X_dev.iloc[itr], y_dev.iloc[itr], g_dev[itr],
                       X_dev.iloc[iva], cfg, num_cols, cat_cols,
                       seeds["model_init_seeds"][0] + 5000 + j, seeds["early_stop_seed"] + 5000 + j)
            oof[cfg["config_id"]][iva] = rec["probs"]
            rows.append({"method": mid, "config_id": cfg["config_id"], "search_type": cfg["search_type"],
                         "inner_fold": j, "inner_AP": float(average_precision_score(y_dev.iloc[iva], rec["probs"])),
                         **{x: rec[x] for x in ["best_epoch", "stopped_epoch", "n_lr_drops", "n_params", "input_dim",
                                                "elapsed_sec", "status", "warnings", "pos_weight_used"]}})
            logs.append(history_frame(rec, method=mid, phase="full_dev_inner", outer_fold=0, inner_fold=j,
                                      config_id=cfg["config_id"]))
    folds = pd.DataFrame(rows)
    assert len(folds) == 36
    agg = (folds.groupby("config_id", sort=False)
           .agg(search_type=("search_type", "first"), mean_AP=("inner_AP", "mean"), sd_AP=("inner_AP", "std"),
                mean_best_epoch=("best_epoch", "mean"), mean_lr_drops=("n_lr_drops", "mean"),
                n_params=("n_params", "median"), total_time_sec=("elapsed_sec", "sum"),
                mean_pos_weight=("pos_weight_used", "mean"))
           .reset_index())
    params = pd.DataFrame(cands)
    params["hidden_layers"] = params["hidden_layers"].astype(str)
    keep = ["config_id", "optimizer", "hidden_layers", "activation", "lr", "weight_decay", "batch_size", "scaler",
            "dropout", "use_scheduler", "clip_outliers", "unknown_strategy"]
    summary = params[keep].merge(agg, on="config_id")
    summary.insert(0, "method", mid)

    def best(stype):
        s = summary[summary.search_type == stype].sort_values(["mean_AP", "sd_AP"], ascending=[False, True])
        return s.iloc[0]
    bm, br = best("manual"), best("random")
    win = br if (br["mean_AP"] > bm["mean_AP"] or (br["mean_AP"] == bm["mean_AP"] and br["sd_AP"] < bm["sd_AP"])) else bm
    cfg = dict(next(c for c in cands if c["config_id"] == win["config_id"]))
    thr = f1_optimal_threshold(y_dev, oof[win["config_id"]]) if meth["threshold_mode"] == "inner_oof_f1" else 0.50
    final = {"method": mid, "method_name": meth["name"], "config": cfg, "config_id": win["config_id"],
             "selected_search": "random_search" if win["search_type"] == "random" else "manual_search",
             "full_dev_mean_AP": float(win["mean_AP"]), "full_dev_sd_AP": float(win["sd_AP"]),
             "best_manual": {"config_id": bm["config_id"], "mean_AP": float(bm["mean_AP"]), "sd_AP": float(bm["sd_AP"])},
             "best_random": {"config_id": br["config_id"], "mean_AP": float(br["mean_AP"]), "sd_AP": float(br["sd_AP"])},
             "threshold": float(thr), "threshold_mode": meth["threshold_mode"], "training_from": tm}
    oof_df = pd.DataFrame({f"oof_{k}": v for k, v in oof.items()})
    return {"folds": folds, "summary": summary.sort_values("mean_AP", ascending=False), "final": final,
            "oof": oof_df, "logs": pd.concat(logs, ignore_index=True)}


# ---------------------------------------------------------------------------
def md_table(df, digits=4):
    """tabulate gerektirmeyen basit Markdown tablo."""
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda v: "" if pd.isna(v) else f"{v:.{digits}f}")
    head = "| " + " | ".join(map(str, d.columns)) + " |"
    sep = "|" + "|".join(["---"] * len(d.columns)) + "|"
    body = ["| " + " | ".join(map(str, r)) + " |" for r in d.astype(str).values]
    return "\n".join([head, sep] + body)
'''
EMBEDDED_FILES['configs_grup/project.json'] = r'''{
  "project_type": "grup",
  "authors": [
    {
      "name": "Süleyman Balsak",
      "student_id": "254312022"
    },
    {
      "name": "Seda Erten",
      "student_id": "254312008"
    },
    {
      "name": "İlkin Urcan",
      "student_id": "244329078"
    },
    {
      "name": "Aslıhan Semiz Varlı",
      "student_id": "254329024"
    }
  ],
  "course": "Yapay Sinir Ağları – Yüksek Lisans Ara Sınav Projesi",
  "instructor": "Gökalp Tulum",
  "seed_owner": "Süleyman Balsak",
  "student_id_last5": "12022",
  "seed_note": "Grup kararıyla temel seed Süleyman Balsak'ın öğrenci numarasından (254312022 → 12022 + 2026 = 14048) türetilmiştir. Dört yöntemin tamamı bu tek seed ve aynı bölünmeyle yürütülmüştür.",
  "experiment_history_note": "Grup daha önce farklı bir kod tabanıyla ve farklı bir seed'le (34104) aynı veri seti üzerinde bir deneme yapmıştır. Bu sürüm yeni seed (14048) ile yeni bir dev/test bölünmesi kullanır ve tüm kararlar yalnızca development verisiyle alınmıştır. Ancak veri seti aynı olduğundan yeni test kümesindeki satırların bir kısmı önceki denemede test olarak değerlendirilmiş olabilir; bu nedenle test sonuçları tam anlamıyla kör kabul edilmemelidir.",
  "method_owners": {
    "A": "Süleyman Balsak",
    "B": "Seda Erten",
    "C": "İlkin Urcan",
    "D": "Aslıhan Semiz Varlı"
  }
}'''

In [ ]:
# === 0. Ortam kurulumu (üç defterde aynı) ===
import os, sys, subprocess, importlib

# Kendi GitHub reponuzun adresi (boş bırakılırsa Drive'daki YSA_Midterm_Project klasörü kullanılır)
GITHUB_REPO_URL = ""   # boş: Drive'daki MyDrive/YSA_Grup_Projesi klasörü kullanılır (önerilen)
PROJECT_SUBDIR = "grup"   # grup projesinin tüm dosyaları (src, configs, results, figures, data) bu klasörde   # ör. "https://github.com/<kullanici-adi>/ysa-mlp-bank-marketing-suleyman-balsak.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if GITHUB_REPO_URL:
        repo = GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
        REPO_DIR = f"/content/drive/MyDrive/{repo}"
        if not os.path.exists(os.path.join(REPO_DIR, ".git")):
            subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)
        else:   # yalnızca kod güncellemelerini çeker; üretilen sonuçlara dokunmaz
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
            print("git pull:", (r.stdout or r.stderr).strip()[:300])
        PROJECT_DIR = os.path.join(REPO_DIR, PROJECT_SUBDIR)   # grup projesi repoda ayrı klasörde
    else:
        PROJECT_DIR = "/content/drive/MyDrive/YSA_Grup_Projesi"
else:  # yerel Jupyter: defter grup/ klasörünün içinde çalıştırılır
    PROJECT_DIR = os.getcwd()

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
# Önceki çalıştırmadan kalan eski klasör adlarını yeni adlara taşı (configs → configs_grup vb.)
for _d in ["configs", "results", "figures", "models", "data", "src"]:
    if os.path.isdir(_d) and not os.path.exists(f"{_d}_grup"):
        os.rename(_d, f"{_d}_grup")
        print(f"Klasör yeniden adlandırıldı: {_d}/ → {_d}_grup/")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for d in ["configs_grup", "results_grup", "results_grup/cache", "figures_grup", "data_grup", "models_grup"]:
    os.makedirs(d, exist_ok=True)

# Gömülü dosyaları yaz (src her seferinde güncellenir; project.json yalnızca yoksa yazılır)
for _p, _t in EMBEDDED_FILES.items():
    if _p == "configs_grup/project.json" and os.path.exists(_p):
        continue
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    with open(_p, "w", encoding="utf-8") as _f:
        _f.write(_t)

if not os.path.exists("src_grup/ysa_utils.py"):
    raise FileNotFoundError(
        f"{PROJECT_DIR}/src/ysa_utils.py bulunamadı!\n"
        "Çözüm 1: GITHUB_REPO_URL'ye kendi repo adresinizi yazın (repo src_grup/ klasörünü içermeli).\n"
        f"Çözüm 2: Drive'da {PROJECT_DIR}/src/ klasörü açıp ysa_utils.py dosyasını oraya yükleyin.")

import src_grup.ysa_utils as U
importlib.reload(U)                      # modül değişirse yeniden yükle
from src_grup.ysa_utils import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Çalışma dizini:", os.getcwd())

---
# BÖLÜM A – Etiketsiz kilitli tahminler (dört yöntem)

In [ ]:
import json, time, joblib
import torch
from IPython.display import Markdown
from src_grup.group_methods import METHODS, METHOD_ORDER, fit_logreg, md_table

SEEDS = load_json("configs_grup/seeds.json")
PROJECT = load_json("configs_grup/project.json")
FINAL = load_json("results_grup/final_hyperparameters.json")
PRIMARY_METHOD = FINAL["primary_method"]
PRIMARY_SEED = SEEDS["model_init_seeds"][0]
torch.set_num_threads(1)

dev_df = pd.read_csv("data_grup/dev_data.csv")
X_dev = prepare_features(dev_df)
y_dev = dev_df[TARGET].astype(int)
g_dev = dev_df[GROUP_COL].values
_test = pd.read_csv("data_grup/test_data.csv")          # etiket sütunu okunduğu anda atılır
test_ids = _test[ID_COL].values
X_test = prepare_features(_test.drop(columns=[TARGET]))
del _test
assert TARGET not in X_test.columns and list(X_test.columns) == list(X_dev.columns)
num_cols, cat_cols = get_feature_lists(X_dev)

display(pd.DataFrame([{"Yöntem": m, "Üye": FINAL["methods"][m]["owner"], "Ad": FINAL["methods"][m]["method_name"],
                       "Nihai aday": FINAL["methods"][m]["config_id"], "Eşik": round(FINAL["methods"][m]["threshold"], 4),
                       "Birincil": "✓" if m == PRIMARY_METHOD else ""} for m in METHOD_ORDER]))
print(f"Birincil: yöntem {PRIMARY_METHOD}, seed {PRIMARY_SEED} | kararlılık seed'leri: {SEEDS['model_init_seeds'][1:]}")

In [ ]:
locked = pd.DataFrame({ID_COL: test_ids})
meta, HIST, trained = [], {}, {}
T0 = time.time()
for mid in METHOD_ORDER:
    f = FINAL["methods"][mid]
    cfg, thr, tm = f["config"], f["threshold"], f["training_from"]
    for i, s in enumerate(SEEDS["model_init_seeds"]):
        key = (tm, f["config_id"], s)
        if key in trained:          # Yöntem C, A ile aynı modeli kullanır → yeniden eğitilmez
            res = trained[key]
        else:
            res = train_mlp(X_dev, y_dev, {"test": X_test}, cfg, num_cols, cat_cols,
                            model_seed=s, es_seed=SEEDS["early_stop_seed"], groups=g_dev)
            trained[key] = res
            tag = f"{tm}_{f['config_id']}_seed{s}"
            torch.save(res["model"].state_dict(), f"models_grup/mlp_{tag}.pt")
            joblib.dump(res["preprocessor"], f"models_grup/preprocessor_{tag}.joblib")
            history_frame(res, method=tm, seed=s).to_csv(f"results_grup/final_training_history_{tag}.csv", index=False)
        p = res["probs"]["test"]
        locked[f"prob_{mid}_seed{s}"] = p
        locked[f"pred_{mid}_seed{s}"] = (p >= thr).astype(int)
        meta.append({"method": mid, "seed": s, "role": "primary" if s == PRIMARY_SEED else "stability",
                     "config_id": f["config_id"], "threshold": thr, "best_epoch": res["best_epoch"],
                     "stopped_epoch": res["stopped_epoch"], "n_params": res["n_params"], "input_dim": res["input_dim"],
                     "pos_weight_used": res["pos_weight_used"], "elapsed_sec": round(res["elapsed_sec"], 2),
                     "status": res["status"], "warnings": res["warnings"], "reused_from": tm if tm != mid else ""})
        if mid == PRIMARY_METHOD and s == PRIMARY_SEED:
            PRIMARY = res
            save_json(get_learned_stats(res["preprocessor"], [c for c in num_cols if not (cfg.get("pdays_mode") == "raw" and c == "prev_contacted")], cat_cols),
                      "models_grup/learned_stats_primary.json")

p_lr = fit_logreg(X_dev, y_dev, X_test, SEEDS, num_cols, cat_cols)
locked["prob_logreg"], locked["pred_logreg"] = p_lr, (p_lr >= 0.5).astype(int)

assert TARGET not in locked.columns
LOCKED_PATH = "results_grup/test_predictions_locked.csv"
locked.to_csv(LOCKED_PATH, index=False)
pd.DataFrame(meta).to_csv("results_grup/final_seeds_train_meta.csv", index=False)
manifest = {"file": LOCKED_PATH, "sha256": file_sha256(LOCKED_PATH), "n_rows": int(len(locked)),
            "contains_labels": False, "primary_method": PRIMARY_METHOD, "primary_seed": PRIMARY_SEED,
            "primary_column": f"prob_{PRIMARY_METHOD}_seed{PRIMARY_SEED}",
            "stability_seeds": SEEDS["model_init_seeds"][1:],
            "thresholds": {m: FINAL["methods"][m]["threshold"] for m in METHOD_ORDER}, "logreg_threshold": 0.5,
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "note": "Test etiketleri okunmadan üretildi. Metrikler yalnızca bu dosyanın commit'inden sonra hesaplanır."}
save_json(manifest, "results_grup/locked_predictions_manifest.json")
print(f"Süre: {time.time() - T0:.0f} sn | SHA-256: {manifest['sha256']}")
display(pd.DataFrame(meta))

### Şekil 4 – Birincil modelin eğitim / iç early-stop kaybı ve LR · Şekil 3 – Mimari

In [ ]:
import matplotlib.patches as mpatches
h = pd.DataFrame(PRIMARY["history"]); be = PRIMARY["best_epoch"]
fcfg = FINAL["methods"][PRIMARY_METHOD]["config"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(h["epoch"], h["train_loss"], lw=2, label="Eğitim (fit) kaybı")
axes[0].plot(h["epoch"], h["es_val_loss"], lw=2, label="İç early-stop BCE")
axes[0].axvline(be, color="red", ls="--", label=f"En iyi epoch = {be}")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Kayıp"); axes[0].legend(fontsize=8)
axes[0].set_title(f"Yöntem {PRIMARY_METHOD} / {FINAL['methods'][PRIMARY_METHOD]['config_id']} – seed {PRIMARY_SEED}")
axes[1].step(h["epoch"], h["lr"], where="post", lw=2, color="#55A868"); axes[1].set_yscale("log")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("LR (log)"); axes[1].set_title(f"ReduceLROnPlateau – düşüş: {PRIMARY['n_lr_drops']}")
plt.tight_layout(); plt.savefig("figures_grup/fig4_loss_and_lr_curve.png", dpi=300); plt.show()

in_dim, hidden = PRIMARY["input_dim"], list(fcfg["hidden_layers"])
dims = [in_dim] + hidden + [1]
layers = pd.DataFrame([{"Katman": f"Linear {i + 1}", "Girdi": dims[i], "Çıktı": dims[i + 1],
                        "Ağırlık": dims[i] * dims[i + 1], "Bias": dims[i + 1]} for i in range(len(dims) - 1)])
layers["Toplam"] = layers["Ağırlık"] + layers["Bias"]
assert layers["Toplam"].sum() == PRIMARY["n_params"] == count_mlp_params(in_dim, hidden)
layers.to_csv("results_grup/final_model_parameter_count.csv", index=False)
display(layers)
W, G = 2.3, 1.1
fig, ax = plt.subplots(figsize=(len(dims) * (W + G) + 0.5, 4))
ax.set_xlim(0, len(dims) * (W + G)); ax.set_ylim(0, 4); ax.axis("off")
for i, d in enumerate(dims):
    x = i * (W + G) + 0.2
    txt = (f"Girdi\n{d} özellik" if i == 0 else
           f"Çıkış\nLinear {dims[-2]}→1\nlogit → σ" if i == len(dims) - 1 else
           f"Gizli {i}\nLinear {dims[i - 1]}→{d}\n{fcfg['activation']}" + (f" + Dropout {fcfg['dropout']}" if fcfg["dropout"] else ""))
    ax.add_patch(mpatches.FancyBboxPatch((x, 1), W, 2.2, boxstyle="round,pad=0.1",
                                         fc=["#DCE6F2", "#FFF2CC", "#FCE4D6"][0 if i == 0 else 2 if i == len(dims) - 1 else 1], ec="#333", lw=1.6))
    ax.text(x + W / 2, 2.1, txt, ha="center", va="center", fontsize=9, fontweight="bold")
    if i < len(dims) - 1:
        ax.annotate("", xy=(x + W + G - 0.18, 2.1), xytext=(x + W + 0.18, 2.1), arrowprops=dict(arrowstyle="-|>", lw=2))
ax.text(len(dims) * (W + G) / 2, 0.4, f"Toplam öğrenilebilir parametre = {PRIMARY['n_params']:,} | {config_label(fcfg)}",
        ha="center", fontsize=8.5, style="italic")
plt.title(f"Şekil 3 – Birincil model mimarisi (Yöntem {PRIMARY_METHOD})", fontweight="bold")
plt.tight_layout(); plt.savefig("figures_grup/fig3_mlp_architecture.png", dpi=300); plt.show()

---
## ⛔ DUR – Kilitli tahminleri GitHub'a commit'leyin
```bash
git add src_grup/ configs_grup/ results_grup/final_hyperparameters.json results_grup/test_predictions_locked.csv \
        results_grup/locked_predictions_manifest.json results_grup/final_seeds_train_meta.csv models_grup/ *.ipynb
git commit -m "Dört yöntemin kilitli test tahminleri (etiketsiz)"
git push
```
Commit hash'ini aşağıya yazıp `LOCK_COMMITTED = True` yapın.

---
# BÖLÜM B – Test etiketleriyle değerlendirme
> Oturum yeniden başladıysa önce **0. Ortam kurulumu** hücresini çalıştırın.

In [ ]:
LOCK_COMMITTED = False        # ← commit'ten sonra True
LOCK_COMMIT_HASH = ""         # ← ör. "3f2a9c1"
assert LOCK_COMMITTED, "Önce test_predictions_locked.csv dosyasını GitHub'a commit'leyin."

import json, time
from IPython.display import Markdown
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix
from src_grup.group_methods import METHODS, METHOD_ORDER, md_table

SEEDS = load_json("configs_grup/seeds.json"); PROJECT = load_json("configs_grup/project.json")
FINAL = load_json("results_grup/final_hyperparameters.json")
manifest = load_json("results_grup/locked_predictions_manifest.json")
assert file_sha256(manifest["file"]) == manifest["sha256"], "Kilitli dosya commit'ten sonra DEĞİŞMİŞ!"
manifest.update(commit_hash=LOCK_COMMIT_HASH, labels_opened_at=time.strftime("%Y-%m-%d %H:%M:%S"))
save_json(manifest, "results_grup/locked_predictions_manifest.json")
PM, PS = manifest["primary_method"], manifest["primary_seed"]
THR = manifest["thresholds"]
locked = pd.read_csv(manifest["file"])
test_full = pd.read_csv("data_grup/test_data.csv")                  # etiketler ŞİMDİ okunur
ev = locked.merge(test_full[[ID_COL, TARGET]], on=ID_COL, how="left", validate="one_to_one")
y_test = ev[TARGET].astype(int).values
OWN = {m: FINAL["methods"][m]["owner"] for m in METHOD_ORDER}
LBL = {m: f"{m} – {METHODS[m]['name']} ({OWN[m]})" for m in METHOD_ORDER}
print(f"Test n = {len(y_test)} | pozitif = {y_test.sum()} (%{100 * y_test.mean():.2f}) | birincil: {PM}, seed {PS}")

### Tablo 6 – Dört yöntemin bağımsız test metrikleri (birincil seed) ve bootstrap %95 GA

In [ ]:
rows = []
for m in METHOD_ORDER:
    p = ev[f"prob_{m}_seed{PS}"].values
    rows.append({"Model": LBL[m], "Yöntem": m, "Birincil": "✓" if m == PM else "", **compute_metrics(y_test, p, THR[m])})
    if THR[m] != 0.5:
        rows.append({"Model": LBL[m] + " @0.50", "Yöntem": m, "Birincil": "", **compute_metrics(y_test, p, 0.5)})
rows.append({"Model": "Logistic Regression", "Yöntem": "LogReg", "Birincil": "", **compute_metrics(y_test, ev["prob_logreg"].values)})
rows.append({"Model": "Dummy (çoğunluk)", "Yöntem": "Dummy", "Birincil": "", **compute_metrics(y_test, np.zeros(len(y_test)))})
test_metrics = pd.DataFrame(rows)
test_metrics.to_csv("results_grup/test_metrics.csv", index=False)
display(test_metrics[["Model", "Birincil", "threshold"] + METRIC_COLS + ["TP", "FP", "TN", "FN"]].round(4))

ci_rows = []
for name, p, t in [(LBL[m], ev[f"prob_{m}_seed{PS}"].values, THR[m]) for m in METHOD_ORDER] + \
                  [("Logistic Regression", ev["prob_logreg"].values, 0.5)]:
    ci, rej = bootstrap_ci(y_test, p, n_boot=1000, seed=SEEDS["bootstrap_seed"], threshold=t)
    for met, d in ci.items():
        ci_rows.append({"Model": name, "Metrik": met, "Nokta": d["point"], "GA_alt": d["ci_low"], "GA_ust": d["ci_high"],
                        "Bootstrap_SD": d["boot_sd"], "n_boot": 1000, "reddedilen_tek_sinif": rej})
ci_tbl = pd.DataFrame(ci_rows)
ci_tbl.to_csv("results_grup/test_bootstrap_ci.csv", index=False)
display(ci_tbl.assign(GA=lambda d: [f"{a:.4f} [{b:.4f}, {c:.4f}]" for a, b, c in zip(d.Nokta, d.GA_alt, d.GA_ust)])
        .pivot(index="Model", columns="Metrik", values="GA"))

### Tablo 7 – Seed kararlılığı (her yöntem, 3 başlangıç seed'i, betimsel)

In [ ]:
srows = []
for m in METHOD_ORDER:
    for s in SEEDS["model_init_seeds"]:
        srows.append({"Yöntem": m, "seed": s, "rol": "birincil" if s == PS else "kararlılık",
                      **compute_metrics(y_test, ev[f"prob_{m}_seed{s}"].values, THR[m])})
seed_df = pd.DataFrame(srows)
seed_df.to_csv("results_grup/seed_stability_results.csv", index=False)
seed_sum = mean_sd_table(seed_df, "Yöntem")
seed_sum.to_csv("results_grup/seed_stability_summary.csv", index=False)
display(seed_sum[["Yöntem", "AP", "ROC_AUC", "F1", "Sensitivity", "Specificity", "Balanced_Accuracy"]])
print("Ana sonuç önceden ilan edilen birincil seed'dir; en yüksek test skoru seçilmemiştir.")

### Şekil 6 – Confusion matrix · Şekil 7 – ROC ve PR eğrileri

In [ ]:
models = [(LBL[m], ev[f"prob_{m}_seed{PS}"].values, THR[m]) for m in METHOD_ORDER] + [("Logistic Regression", ev["prob_logreg"].values, 0.5)]
fig, axes = plt.subplots(1, len(models), figsize=(4.3 * len(models), 4))
for ax, (name, p, t) in zip(axes, models):
    cm = confusion_matrix(y_test, (p >= t).astype(int), labels=[0, 1])
    pct = cm / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm, annot=np.array([[f"{cm[i, j]}\n(%{pct[i, j]:.1f})" for j in range(2)] for i in range(2)]), fmt="",
                cmap="Blues", cbar=False, ax=ax, xticklabels=["T: no", "T: yes"], yticklabels=["G: no", "G: yes"])
    ax.set_title(name.split(" (")[0] + f"\neşik = {t:.3f}" + ("  [BİRİNCİL]" if name.startswith(PM + " ") else ""), fontsize=9)
plt.suptitle("Şekil 6 – Bağımsız test confusion matrix (satır yüzdesi)", fontweight="bold")
plt.tight_layout(); plt.savefig("figures_grup/fig6_confusion_matrices.png", dpi=300); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for (name, p, t), c in zip(models, sns.color_palette("tab10", len(models))):
    m = compute_metrics(y_test, p, t)
    fpr, tpr, _ = roc_curve(y_test, p); prec, rec, _ = precision_recall_curve(y_test, p)
    lw = 3 if name.startswith(PM + " ") else 1.6
    axes[0].plot(fpr, tpr, color=c, lw=lw, label=f"{name.split(' (')[0]}  AUC={m['ROC_AUC']:.3f}")
    axes[1].step(rec, prec, where="post", color=c, lw=lw, label=f"{name.split(' (')[0]}  AP={m['AP']:.3f}")
    axes[0].scatter(1 - m["Specificity"], m["Sensitivity"], color=c, edgecolor="black", zorder=5)
    axes[1].scatter(m["Sensitivity"], m["Precision"], color=c, edgecolor="black", zorder=5)
axes[0].plot([0, 1], [0, 1], "k:"); axes[1].axhline(y_test.mean(), color="gray", ls=":")
axes[0].set_xlabel("1 − Specificity"); axes[0].set_ylabel("Sensitivity"); axes[0].set_title("ROC – test")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision–Recall – test")
for ax in axes: ax.legend(fontsize=7)
plt.suptitle("Şekil 7 – Dört yöntem ve LogReg (kalın çizgi = birincil; nokta = seçilen eşikteki çalışma noktası)", fontweight="bold")
plt.tight_layout(); plt.savefig("figures_grup/fig7_roc_pr_curves.png", dpi=300); plt.show()

### Hata analizi – birincil yöntem (Bölüm 13.2)

In [ ]:
p = ev[f"prob_{PM}_seed{PS}"].values; pred = (p >= THR[PM]).astype(int)
grp = np.select([(y_test == 1) & (pred == 1), (y_test == 0) & (pred == 1), (y_test == 1) & (pred == 0)], ["TP", "FP", "FN"], "TN")
ea = test_full.set_index(ID_COL).loc[ev[ID_COL]].reset_index().assign(group=grp, prob=p)
err = ea.groupby("group").agg(n=("prob", "size"), ort_olasilik=("prob", "mean"), yas_medyan=("age", "median"),
                              campaign_medyan=("campaign", "median"), euribor3m_medyan=("euribor3m", "median"),
                              onceki_temas_pct=("pdays", lambda x: 100 * (x != PDAYS_NO_CONTACT).mean()),
                              poutcome_success_pct=("poutcome", lambda x: 100 * (x == "success").mean()),
                              cellular_pct=("contact", lambda x: 100 * (x == "cellular").mean())).reindex(["TP", "FN", "FP", "TN"]).round(3)
err.to_csv("results_grup/error_analysis_groups.csv"); display(err)
fn = ea[ea.group == "FN"]
print(f"FN = {len(fn)} (pozitiflerin %{100 * len(fn) / max(1, y_test.sum()):.1f}'i) | FP = {(grp == 'FP').sum()} | "
      f"FN'lerin {int((fn.prob >= THR[PM] * 0.6).sum())} tanesi eşiğe yakın")

### Üç seviyenin ayrı yorumu: dış CV (prosedür) – development iç CV (seçim) – bağımsız test

In [ ]:
outer_sum = pd.read_csv("results_grup/outer_cv_summary_mean_sd.csv").set_index("grp")
tm = test_metrics.drop_duplicates("Yöntem").set_index("Yöntem")
lv = []
for m in METHOD_ORDER:
    f = FINAL["methods"][m]
    og = f"MLP-{m} {'random' if f['selected_search'] == 'random_search' else 'manual'}"
    lv.append({"Yöntem": m, "Üye": OWN[m], "Dev iç CV AP (seçim)": f"{f['full_dev_mean_AP']:.4f} ± {f['full_dev_sd_AP']:.4f}",
               "Dış CV AP (seçilen arama)": outer_sum.loc[og, "AP"], "Dış CV F1": outer_sum.loc[og, "F1"],
               "Test AP": round(tm.loc[m, "AP"], 4), "Test F1": round(tm.loc[m, "F1"], 4),
               "Test Sensitivity": round(tm.loc[m, "Sensitivity"], 4), "Birincil": "✓" if m == PM else ""})
lv.append({"Yöntem": "LogReg", "Üye": "-", "Dev iç CV AP (seçim)": "-", "Dış CV AP (seçilen arama)": outer_sum.loc["LogReg", "AP"],
           "Dış CV F1": outer_sum.loc["LogReg", "F1"], "Test AP": round(tm.loc["LogReg", "AP"], 4),
           "Test F1": round(tm.loc["LogReg", "F1"], 4), "Test Sensitivity": round(tm.loc["LogReg", "Sensitivity"], 4), "Birincil": ""})
levels = pd.DataFrame(lv)
levels.to_csv("results_grup/methods_three_level_comparison.csv", index=False)
display(levels)

### Şekil 1 – Deneysel tasarım akış şeması

In [ ]:
import matplotlib.patches as mpatches
ss = pd.read_csv("results_grup/dataset_split_summary.csv").set_index("Küme")
n_all, n_dev, n_te = (int(ss.loc[k, "n"]) for k in ["Tüm veri", "Development (%80)", "Independent Test (%20)"])
cp = ci_tbl[ci_tbl.Model == LBL[PM]].set_index("Metrik")
def box(ax, x, y, w, h, t, fc, ec, fs=8.6):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12", fc=fc, ec=ec, lw=1.8))
    ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=fs, fontweight="bold")
fig, ax = plt.subplots(figsize=(13, 7.4)); ax.set_xlim(0, 13); ax.set_ylim(0, 7.6); ax.axis("off")
box(ax, 3.5, 6.6, 6, .75, f"bank-additional.csv (n = {n_all:,}) · 'duration' çıkarıldı · seed {SEEDS['base_seed']} ({PROJECT['seed_owner']})", "#DCE6F2", "#2B5B84")
box(ax, .4, 5.15, 6, .85, f"Development (%80, n = {n_dev:,})\nEDA · 4 yöntem × nested CV · nihai eğitim", "#E2EFDA", "#385723")
box(ax, 7, 5.15, 5.6, .85, f"Bağımsız test (%20, n = {n_te:,})\nkopya grubuna duyarlı, kilitli", "#FCE4D6", "#C55A11")
box(ax, .4, 3.25, 12.2, 1.4, "Aynı dış (3) ve iç (3) katlar · aynı 6 manuel + 6 otomatik aday · her yöntem: 108 iç eğitim + 6 refit + 36 tam-dev\n"
    + "   ".join(f"{m}: {METHODS[m]['name']} ({OWN[m].split()[0]})" for m in METHOD_ORDER), "#FFF2CC", "#BF9000", fs=8.2)
box(ax, .4, 1.75, 6, .95, f"Birincil yöntem (önceden tanımlı kural, dev iç CV AP): {PM}\n"
    f"{FINAL['methods'][PM]['config_id']} – iç AP = {FINAL['methods'][PM]['full_dev_mean_AP']:.4f}", "#D9E1F2", "#2F5597")
box(ax, 7, 1.75, 5.6, .95, f"4 yöntem × 3 seed {SEEDS['model_init_seeds']} + LogReg\n→ tek kilitli dosya → ayrı commit", "#E2EFDA", "#385723")
box(ax, 2, .2, 9, .9, f"Test (birincil {PM}, seed {PS}): AP = {cp.loc['AP', 'Nokta']:.4f} [{cp.loc['AP', 'GA_alt']:.4f}, {cp.loc['AP', 'GA_ust']:.4f}]"
    f" | ROC-AUC = {cp.loc['ROC_AUC', 'Nokta']:.4f} | F1 = {cp.loc['F1', 'Nokta']:.4f} (1.000 bootstrap)", "#F4CCCC", "#990000")
for (a, b), (c, d) in [((5.5, 6.58), (3.4, 6.05)), ((7.5, 6.58), (9.8, 6.05)), ((3.4, 5.13), (3.4, 4.7)),
                       ((3.4, 3.23), (3.4, 2.75)), ((6.45, 2.22), (6.95, 2.22)), ((9.8, 1.73), (8, 1.15))]:
    ax.annotate("", xy=(c, d), xytext=(a, b), arrowprops=dict(arrowstyle="->", lw=1.6))
plt.title("Şekil 1 – Veri bölünmesi, dört yöntem ve nested CV", fontweight="bold")
plt.tight_layout(); plt.savefig("figures_grup/fig1_nested_cv_flow.png", dpi=300); plt.show()

### README, requirements ve rapor taslağı (tüm sayılar dosyalardan)

In [ ]:
sysinfo = load_json("results_grup/system_info.json")
with open("requirements.txt", "w") as f:
    f.write("\n".join([f"numpy=={sysinfo['numpy_version']}", f"pandas=={sysinfo['pandas_version']}",
                       f"scikit-learn=={sysinfo['sklearn_version']}", f"torch=={sysinfo['torch_version'].split('+')[0]}",
                       f"matplotlib=={sysinfo['matplotlib_version']}", f"seaborn=={sysinfo['seaborn_version']}", "scipy", "joblib"]) + "\n")
with open(".gitignore", "w") as f:
    f.write("data_grup/\nresults/cache/\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n")

AUTH = ", ".join(f"{a['name']} ({a['student_id']})" for a in PROJECT["authors"])
meth_tbl = pd.DataFrame([{"Yöntem": m, "Üye": OWN[m], "Ad": METHODS[m]["name"], "Nasıl": METHODS[m]["how"],
                          "Nihai aday": FINAL["methods"][m]["config_id"], "Eşik": round(THR[m], 3)} for m in METHOD_ORDER])
cpm = ci_tbl[ci_tbl.Model == LBL[PM]].set_index("Metrik")
readme = f"""# Bank Marketing – MLP ile Sınıflandırma: Dört Yöntem, Nested CV ve Bağımsız Test

**Hazırlayanlar:** {AUTH}
**Ders:** {PROJECT['course']} · **Ders sorumlusu:** {PROJECT.get('instructor', '')}
**Temel seed:** `{SEEDS['base_seed']}` (= {SEEDS['student_last5']} + 2026; kaynak: {PROJECT['seed_owner']})

## Veri
UCI Bank Marketing, yalnızca `bank-additional.csv` (4.119 gözlem). Moro, Rita ve Cortez (2014), https://doi.org/10.24432/C5K306.
Ham veri repoya eklenmez; `grup01_eda.ipynb` resmî bağlantıdan indirir. `duration` tüm modellerden çıkarılmıştır.

## Çalıştırma (Google Colab)
Defterlerin ilk hücresindeki `GITHUB_REPO_URL` bu repoyu gösterir; Colab repoyu Drive'a klonlar.
1. `grup01_eda.ipynb` – seed'ler, indirme, kopya grupları, %80/%20 bölünme, development-only EDA, hipotezler
2. `grup02_nested_cv.ipynb` – her üyenin yöntemi (A–D) için 3×3 nested CV ve tam-dev seçimi; birincil yöntem kuralı
3. `grup03_final_test.ipynb` – Bölüm A: 4 yöntem × 3 seed, kilitli tahmin → *ayrı commit* → Bölüm B: test, GA, şekiller, rapor taslağı

Ortak kod: `src_grup/ysa_utils.py`, yöntemler: `src_grup/group_methods.py`.

## Dört yöntem
{md_table(meth_tbl)}

## Temel sonuçlar
- Birincil yöntem (önceden tanımlı kural): **{PM} – {METHODS[PM]['name']}**, aday {FINAL['methods'][PM]['config_id']}
- Bağımsız test (seed {PS}): AP = {cpm.loc['AP', 'Nokta']:.4f} [{cpm.loc['AP', 'GA_alt']:.4f}, {cpm.loc['AP', 'GA_ust']:.4f}],
  ROC-AUC = {cpm.loc['ROC_AUC', 'Nokta']:.4f}, F1 = {cpm.loc['F1', 'Nokta']:.4f}
- Tüm yöntemler: `results_grup/methods_three_level_comparison.csv`, `results_grup/test_metrics.csv`
- Kilitli tahmin commit'i: `{LOCK_COMMIT_HASH or 'BELİRTİLMEDİ'}` · SHA-256 `{manifest['sha256'][:16]}…`

## Sistem
{sysinfo['environment']} · {sysinfo['os']} · Python {sysinfo['python_version']} · PyTorch {sysinfo['torch_version']} · CPU, tek thread
""" + (f"\n## Deney geçmişi\n{PROJECT['experiment_history_note']}\n" if PROJECT.get("experiment_history_note") else "")
open("README.md", "w", encoding="utf-8").write(readme)
print("README.md yazıldı.")

In [ ]:
split_s = pd.read_csv("results_grup/dataset_split_summary.csv")
dup = pd.read_csv("results_grup/duplicate_check.csv")
outer_tbl = pd.read_csv("results_grup/outer_cv_summary_mean_sd.csv")[["grp", "AP", "ROC_AUC", "F1", "Sensitivity", "Specificity", "Balanced_Accuracy"]]
test_tbl = test_metrics[["Model", "threshold", "AP", "ROC_AUC", "F1", "Sensitivity", "Specificity", "Balanced_Accuracy"]]
ci_piv = ci_tbl.assign(GA=lambda d: [f"{a:.4f} [{b:.4f}, {c:.4f}]" for a, b, c in zip(d.Nokta, d.GA_alt, d.GA_ust)]) \
               .pivot(index="Model", columns="Metrik", values="GA").reset_index()
seed_tbl = seed_sum[["Yöntem", "AP", "ROC_AUC", "F1"]]

def method_section(m):
    s = pd.read_csv(f"results_grup/method_{m}/full_dev_search_summary.csv")
    cols = ["config_id", "optimizer", "hidden_layers", "lr", "scaler", "mean_AP", "sd_AP", "mean_best_epoch", "total_time_sec"]
    f = FINAL["methods"][m]
    return f"""### 5.{METHOD_ORDER.index(m) + 1} Yöntem {m} – {METHODS[m]['name']} (Sorumlu: {OWN[m]})
**Yöntem:** {METHODS[m]['how']}
**Hipotez:** {METHODS[m]['hypothesis']}

**Tablo 2-{m} / 3-{m}.** Manuel ve otomatik adayların tam development iç CV sonuçları:

{md_table(s[cols])}

Nihai aday: **{f['config_id']}** ({f['selected_search']}), iç CV AP = {f['full_dev_mean_AP']:.4f} ± {f['full_dev_sd_AP']:.4f}, eşik = {f['threshold']:.3f}. Şekil 5-{m}: `figures_grup/fig5{m}_method_{m}_search.png`.

**Yorum ({OWN[m]}):** _grup02_nested_cv.ipynb'deki "Yorum – {OWN[m]}" bölümündeki yanıtı buraya aktarın._
"""

rapor = f"""# MLP ile Sınıflandırma: EDA, Ön İşleme, Hiperparametre Optimizasyonu ve Nested Cross-Validation
**{PROJECT['course']}** · Hazırlayanlar: {AUTH} · Ders sorumlusu: {PROJECT.get('instructor', '')}

> Bu dosya `grup03_final_test.ipynb` tarafından otomatik üretilmiş **taslaktır**. Tüm sayılar `results_grup/` dosyalarından gelir. _İtalik_ yer tutucuları grup kendi cümleleriyle doldurmalıdır. Hedef uzunluk 8–12 sayfa (Bölüm 16).

## Özet ve araştırma sorusu
_Problem, veri, yaklaşım ve temel bulguyu 120–150 kelimede yazın._ Anahtar sayılar: birincil yöntem **{PM} ({METHODS[PM]['name']})**, bağımsız test AP = {cpm.loc['AP', 'Nokta']:.4f} [%95 GA {cpm.loc['AP', 'GA_alt']:.4f}–{cpm.loc['AP', 'GA_ust']:.4f}], ROC-AUC = {cpm.loc['ROC_AUC', 'Nokta']:.4f}; Logistic Regression test AP = {tm.loc['LogReg', 'AP']:.4f}; pozitif oranı = {y_test.mean():.4f}.
**Araştırma sorusu:** Dengesiz bir ikili sınıflandırma probleminde MLP'nin performansı ön işleme, optimizer, eğitim kaybı ve karar kuralı tercihlerine ne kadar duyarlıdır ve bu tercihler doğrusal bir referansa göre fark yaratır mı?

## 1. Giriş
_MLP, hiperparametre optimizasyonu, sınıf dengesizliği ve nested CV'nin neden gerekli olduğu; çalışmanın amacı ve dört yöntemin neden seçildiği._

## 2. Veri seti ve EDA
{md_table(split_s)}

Kopya kontrolü:

{md_table(dup)}

`duration` görüşme öncesinde bilinmediği için çıkarılmıştır. `pdays = 999` kodu `prev_contacted` göstergesine ayrılmıştır (Yöntem D'de ham bırakılır). Şekil 2a–2g: `figures_grup/fig2*.png`. _EDA bulgularını ve 01'deki hipotezleri (H1–H6) özetleyin._

## 3. Yöntem
- **Bölünme ve seed:** temel seed {SEEDS['base_seed']} ({PROJECT['seed_owner']}); alt seed'ler `configs_grup/seeds.json`. Tüm bölünmeler kopya grubuna duyarlıdır (Şekil 1).
- **MLP:** PyTorch, tek logit + BCEWithLogitsLoss, ReLU/tanh, en fazla 150 epoch, early stopping (patience 15, en iyi ağırlıklar), ReduceLROnPlateau (factor 0.5, patience 5, min_lr 1e-5), iç early-stop kümesi %15. Birincil modelin parametre sayısı: {int(pd.read_csv('results_grup/final_model_parameter_count.csv')['Toplam'].sum()):,} (Şekil 3).
- **Kat-içi ön işleme:** median imputation → (opsiyonel) P1–P99 clipping → Standard/MinMax/Robust scaler; kategorikler için `unknown` stratejisi → OneHotEncoder(handle_unknown="ignore"). Standardization (z-skoru) ile normalization (MinMax aralık ölçekleme) farkı _açıklanmalı_.
- **Nested CV:** dış 3 kat / iç 3 kat; 6 manuel (M1–M6) + 6 otomatik (R1–R6, optimizer başına 2) aday; seçim iç CV ortalama AP ile.
- **Dört yöntem (üye iş bölümü):**

{md_table(meth_tbl)}

- **Birincil yöntem kuralı:** {FINAL['primary_rule']}.

## 4. Referans modeller
Dummy (çoğunluk sınıfı) ve Logistic Regression (sabit ön işleme) aynı dış katlarda değerlendirilmiştir (Tablo 5).

## 5. Deneyler – dört yöntem
{''.join(method_section(m) for m in METHOD_ORDER)}

## 6. Nested CV bulguları
**Tablo 5.** Dış CV Mean ± SD (n = 3; katlar bağımsız tekrar değildir, SD güven aralığı değildir):

{md_table(outer_tbl)}

Şekil 5: `figures_grup/fig5_methods_outer_cv.png`. _Yöntemleri ve referansları adil biçimde karşılaştırın._

## 7. Bağımsız test
**Tablo 6.** Test metrikleri (birincil seed {PS}):

{md_table(test_tbl)}

**Bootstrap %95 GA (1.000 tekrar):**

{md_table(ci_piv)}

**Tablo 7.** Seed kararlılığı (Mean ± SD, n = 3):

{md_table(seed_tbl)}

Şekil 6–7: `figures_grup/fig6_confusion_matrices.png`, `figures_grup/fig7_roc_pr_curves.png`. Hata analizi: `results_grup/error_analysis_groups.csv`.

## 8. Tartışma
**Üç seviyenin karşılaştırması:**

{md_table(levels)}

_Her üye kendi yönteminin etkisini 3–4 cümleyle tartışır; ardından ortak değerlendirme:_ hiperparametre etkisi, optimizer farkları, sınıf ağırlığı ve eşik seçiminin pratik anlamı, ön işleme duyarlılığı, MLP–LogReg farkı.
**Sınırlamalar:** n = 3 dış kat; küçük test kümesi ({len(y_test)} gözlem, {int(y_test.sum())} pozitif); müşteri kimliği olmadığı için kişi-bağımsız genelleme kanıtlanamaz; kamuya açık veri nedeniyle kilitli commit kör test garantisi değildir.{(' ' + PROJECT['experiment_history_note']) if PROJECT.get('experiment_history_note') else ''}

## 9. Sonuç
_Kanıta dayalı 4–6 cümle._

## Kaynakça
1. Moro, S., Rita, P., Cortez, P. (2014). Bank Marketing. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306
2. scikit-learn. Common pitfalls and recommended practices. https://scikit-learn.org/stable/common_pitfalls.html
3. PyTorch. ReduceLROnPlateau. https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html
4. scikit-learn. Cross-validation, model selection and classification metrics. https://scikit-learn.org/stable/

## Ek – Üye katkıları
{md_table(pd.DataFrame([{"Üye": OWN[m], "Yöntem": f"{m} – {METHODS[m]['name']}", "Defter bölümü": f"grup02_nested_cv.ipynb, Bölüm {METHOD_ORDER.index(m) + 2}", "Çıktılar": f"results_grup/method_{m}/, Tablo 2-{m}…5-{m}, Şekil 5-{m}"} for m in METHOD_ORDER]))}
Ortak bölümler (EDA, test değerlendirmesi, rapor) grup tarafından birlikte yürütülmüştür.
"""
open("rapor_taslagi.md", "w", encoding="utf-8").write(rapor)
print(f"rapor_taslagi.md yazıldı ({len(rapor.split()):,} kelime). Word/Google Docs'a aktarıp PDF olarak dışa aktarabilirsiniz.")